In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorSampler

from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import TrainableFidelityQuantumKernel
from qiskit_machine_learning.algorithms import QuantumKernelTrainer
from qiskit_algorithms.optimizers import SPSA

# ==========================
# LOAD DATA
# ==========================

dataset = pd.read_csv("../dataset/riemann_features.csv")

features = [
"z_co_gram_lag_2",
"z_gram",
"z_gram_lag_1",
"d_lag_13",
"z_co_gram_lag_3",
"z_co_gram_lag_1",
"d_lag_14",
"d_lag_1",
"z_gram_lag_2",
"d_lag_17"
]

X = dataset[features].values
y = dataset["distance"].values

X = X[:2000]
y = y[:2000]

split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

# ==========================
# SCALE
# ==========================

scaler = MinMaxScaler(feature_range=(-1,1))

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================
# PCA
# ==========================

pca = PCA(n_components=5)

X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

n_qubits = X_train.shape[1]

# ==========================
# QUANTUM NEURAL FEATURE MAP
# ==========================

x = ParameterVector("x", n_qubits)
theta = ParameterVector("θ", n_qubits * 3)

qc = QuantumCircuit(n_qubits)

# encoding layer
for i in range(n_qubits):
    qc.ry(x[i], i)

# variational layers
k = 0

for layer in range(3):

    for i in range(n_qubits):

        qc.rz(theta[k], i)
        k += 1

    for i in range(n_qubits - 1):

        qc.cx(i, i + 1)

# ==========================
# KERNEL
# ==========================

sampler = StatevectorSampler()

fidelity = ComputeUncompute(sampler=sampler)

quantum_kernel = TrainableFidelityQuantumKernel(
    feature_map=qc,
    fidelity=fidelity,
    training_parameters=theta
)

# ==========================
# TRAIN KERNEL
# ==========================

optimizer = SPSA(maxiter=50)

kernel_trainer = QuantumKernelTrainer(
    quantum_kernel=quantum_kernel,
    optimizer=optimizer
)

print("Training Quantum Neural Kernel...")

kernel_result = kernel_trainer.fit(X_train, y_train)

optimized_kernel = kernel_result.quantum_kernel

# ==========================
# SVR
# ==========================

svr = SVR(
    kernel=optimized_kernel.evaluate,
    C=20,
    epsilon=0.0005
)

svr.fit(X_train, y_train)

pred = svr.predict(X_test)

# ==========================
# METRICS
# ==========================

rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\n===== Quantum Neural Kernel =====")
print("RMSE:", rmse)
print("R2:", r2)